# "Speak with the PDF" Mini App (RAG)
**Kunal Ahuja** | Automate with GenAI & Agents | Hands-On Activity HO1

Pipeline: PDF -> text -> overlapping chunks -> embeddings (OpenRouter) -> in-memory vector index (NumPy cosine similarity) -> top-k retrieval -> grounded answer from the LLM (OpenRouter).
The sample document is a fictional handbook (`Harbor Point Solar Handbook`), so a vanilla LLM cannot know its facts. That makes the hallucination vs. grounding comparison clear.

**Run order:** Run all cells. When Cell 4 asks, upload `Harbor_Point_Solar_Handbook.pdf`. The API key is read from Colab Secrets (`OPENROUTER_API_KEY`) or a hidden prompt and is never stored in the notebook.

## Cell 1: Install dependencies and load the API key securely

In [8]:
%pip install -q openai pypdf numpy pandas

import time
from getpass import getpass

import numpy as np
import pandas as pd
from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("OPENROUTER_API_KEY")   # Colab Secrets (key icon)
except Exception:
    api_key = None
if not api_key:
    api_key = getpass("Enter your OpenRouter API key: ")

api_key = (api_key or "").strip()
print(len(api_key), api_key.startswith("sk-or-"))

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)


Enter your OpenRouter API key: ··········
73 True


## Cell 2: Configuration

In [9]:
LLM_MODEL = "openai/gpt-4o-mini"              # answers questions
EMBED_MODEL = "openai/text-embedding-3-small"  # embeds chunks + queries
# If a slug returns 404, pick a current one from https://openrouter.ai/models
# and change the string here; nothing else in the code needs to change.

CHUNK_SIZE = 800      # characters per chunk
CHUNK_OVERLAP = 150   # characters shared between neighbouring chunks
TOP_K = 4             # chunks retrieved per question

## Cell 3: Chunking

In [10]:
def chunk_text(text: str, chunk_size: int, overlap: int) -> list:
    """
    Split `text` into overlapping chunks of roughly `chunk_size` characters.
    Overlap keeps a sentence that straddles a boundary intact in at least
    one chunk. Chunks end at a space where possible so words aren't cut.
    """
    text = " ".join(text.split())          # normalise whitespace
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):                # back up to the last space
            space = text.rfind(" ", start, end)
            if space > start + chunk_size // 2:
                end = space
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = end - overlap              # step back for the overlap
    return [c for c in chunks if c]

## Cell 4: Upload the PDF and extract the raw text

In [11]:
from pypdf import PdfReader

# Works in Colab (upload dialog) and in local Jupyter (reads the file from the
# notebook's folder).
PDF_FILE = "Harbor_Point_Solar_Handbook.pdf"
try:
    from google.colab import files
    uploaded = files.upload()               # choose Harbor_Point_Solar_Handbook.pdf
    pdf_path = list(uploaded.keys())[0]
except ImportError:
    pdf_path = PDF_FILE                     # local: keep the PDF next to this notebook

reader = PdfReader(pdf_path)
pages = [page.extract_text() or "" for page in reader.pages]
full_text = "\n".join(pages)

print(f"Loaded '{pdf_path}': {len(reader.pages)} pages, {len(full_text):,} characters")
print("Preview:", full_text[:300].replace("\n", " "))

Saving Harbor_Point_Solar_Handbook.pdf to Harbor_Point_Solar_Handbook (1).pdf
Loaded 'Harbor_Point_Solar_Handbook (1).pdf': 2 pages, 4,890 characters
Preview: Harbor Point Solar Cooperative Operations Handbook 2025 This handbook describes how the Harbor Point Solar Cooperative runs its generation site, maintains its equipment, responds to faults and outages, and reports to its members. It is written for field technicians, control room staff and members of


## Cell 5: Embeddings and in-memory vector index

In [12]:
def embed_texts(texts: list, batch_size: int = 64) -> np.ndarray:
    """Embed a list of strings via OpenRouter; returns an (n, dim) array."""
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        vectors.extend([d.embedding for d in resp.data])
    return np.array(vectors, dtype=np.float32)

def build_index(text: str, chunk_size: int, overlap: int) -> dict:
    """Chunk the text, embed every chunk, and store it all in one dict."""
    chunks = chunk_text(text, chunk_size, overlap)
    vecs = embed_texts(chunks)
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)   # unit length, so
    return {"chunks": chunks, "vectors": vecs,            # dot product =
            "chunk_size": chunk_size, "overlap": overlap}  # cosine similarity

index = build_index(full_text, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Indexed {len(index['chunks'])} chunks (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})")

Indexed 8 chunks (size=800, overlap=150)


## Cell 6: Retrieval (vector search)

In [13]:
def retrieve(question: str, idx: dict, k: int = TOP_K) -> list:
    """Embed the question and return the k most similar chunks + scores."""
    q = embed_texts([question])[0]
    q /= np.linalg.norm(q)
    scores = idx["vectors"] @ q
    top = np.argsort(scores)[::-1][:k]
    return [{"chunk": idx["chunks"][i], "score": float(scores[i])} for i in top]

## Cell 7: Vanilla LLM vs RAG-augmented LLM

In [14]:
def ask_llm(messages: list, temperature: float = 0.2) -> dict:
    """Single LLM call through OpenRouter; returns text, latency, tokens."""
    start = time.time()
    resp = client.chat.completions.create(
        model=LLM_MODEL, messages=messages,
        temperature=temperature, max_tokens=500)
    return {"answer": resp.choices[0].message.content,
            "latency_sec": round(time.time() - start, 2),
            "total_tokens": resp.usage.total_tokens}

def vanilla_answer(question: str) -> dict:
    """No retrieval: the model answers from its own training data only."""
    return ask_llm([{"role": "user", "content": question}])

def rag_answer(question: str, idx: dict, k: int = TOP_K) -> dict:
    """Retrieve relevant chunks, then ask the LLM to answer from them only."""
    start = time.time()
    hits = retrieve(question, idx, k)
    context = "\n\n".join(f"[Chunk {i+1}] {h['chunk']}" for i, h in enumerate(hits))
    system = ("Answer the question using ONLY the context from the PDF below. "
              "If the context does not contain the answer, say so instead of "
              "guessing.\n\nCONTEXT:\n" + context)
    result = ask_llm([{"role": "system", "content": system},
                      {"role": "user", "content": question}])
    result["latency_sec"] = round(time.time() - start, 2)  # includes retrieval
    result["hits"] = hits
    return result

## Cell 8: Chunk-size experiment

In [15]:
EXPERIMENT_QUESTION = "At what temperature does an inverter showing fault code E-17 shut down?"

for size, overlap in [(300, 50), (800, 150), (1500, 250)]:
    exp_index = build_index(full_text, size, overlap)
    top = retrieve(EXPERIMENT_QUESTION, exp_index, k=1)[0]
    print(f"\n--- chunk_size={size} | {len(exp_index['chunks'])} chunks | top score={top['score']:.3f} ---")
    print(top["chunk"][:400], "...")


--- chunk_size=300 | 20 chunks | top score=0.800 ---
sing numeric codes. Technicians should acknowledge every fault in the control system before starting any physical work. Code E-17 indicates inverter overheating. An inverter reporting E-17 shuts down automatically when its internal temperature reaches 78 degrees Celsius, and it restarts only after ...

--- chunk_size=800 | 8 chunks | top score=0.807 ---
7 indicates inverter overheating. An inverter reporting E-17 shuts down automatically when its internal temperature reaches 78 degrees Celsius, and it restarts only after cooling below 65 degrees Celsius and being manually reset by a technician. Code E-42 indicates a grid frequency deviation. It is raised when the grid frequency moves more than 0.5 Hz away from nominal, and the affected inverters  ...

--- chunk_size=1500 | 4 chunks | top score=0.686 ---
d acknowledge every fault in the control system before starting any physical work. Code E-17 indicates inverter overheating. An inv

## Cell 9: Three test questions, vanilla vs RAG

In [16]:
QUESTIONS = [
    "What does fault code E-17 mean, and at what temperature does the inverter shut down?",
    "Which facilities get priority power restoration during an outage, and how quickly must power be restored to them?",
    "How much electricity did the cooperative generate in 2025, and what was the capacity factor?",
]

rows = []
for q in QUESTIONS:
    v = vanilla_answer(q)
    r = rag_answer(q, index)
    print(f"\n{'='*70}\nQUESTION: {q}\n{'='*70}")
    print(f"\n--- VANILLA LLM ({v['latency_sec']}s, {v['total_tokens']} tokens) ---")
    print(v["answer"])
    print(f"\n--- RAG LLM ({r['latency_sec']}s, {r['total_tokens']} tokens) ---")
    print(r["answer"])
    print("\nTop retrieved chunks (similarity score):")
    for h in r["hits"]:
        print(f"  [{h['score']:.3f}] {h['chunk'][:120]}...")
    rows.append({"question": q,
                 "vanilla_answer": v["answer"], "vanilla_sec": v["latency_sec"],
                 "vanilla_tokens": v["total_tokens"],
                 "rag_answer": r["answer"], "rag_sec": r["latency_sec"],
                 "rag_tokens": r["total_tokens"]})

pd.DataFrame(rows).to_csv("rag_comparison_results.csv", index=False)
print("\nSaved to rag_comparison_results.csv")


QUESTION: What does fault code E-17 mean, and at what temperature does the inverter shut down?

--- VANILLA LLM (2.19s, 164 tokens) ---
The fault code E-17 typically indicates an over-temperature condition in the inverter of a solar inverter system. This means that the inverter has detected that its internal temperature has exceeded safe operating limits, prompting it to shut down to prevent damage.

The specific temperature at which the inverter shuts down can vary by manufacturer and model, but it is commonly around 85°C (185°F). However, for precise information, it's best to consult the user manual or technical documentation for your specific inverter model, as different models may have different thresholds and safety features. If you're experiencing this fault code, it's advisable to check the inverter's cooling system, ensure proper ventilation, and possibly contact technical support for further assistance.

--- RAG LLM (1.65s, 747 tokens) ---
Fault code E-17 indicates inverter o